# Metacognitive prompting

Effective problem-solving requires not just thinking about a problem, but thinking about *how* to think about it. A good solver does not blindly apply the same approach to every challenge - they first assess the problem's nature, choose a strategy suited to it, check whether that strategy is working, and switch if it is not. This self-awareness about one's own reasoning process is called metacognition, and it is what separates adaptive problem-solving from rote application of a single technique.

Metacognitive prompting brings this control loop to LLM reasoning. Instead of sending a problem directly to the model and accepting whatever comes back, the system first asks the model to reason about what *kind* of problem this is and which reasoning approach it calls for. It then applies that approach, evaluates whether the result is complete and correct, and - if not - adapts by switching to a better-suited strategy. The model is doing two things: solving at the object level and supervising its own solving at the meta level.

The core insight is that strategy selection is itself a reasoning task, and LLMs are capable of it. Asking "how should I approach this?" before "here is my answer" tends to produce more thoughtful, better-structured solutions - especially for problems where the wrong approach yields a confident but shallow answer.

In [1]:
import os
from typing import List, Tuple

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

### Initialize the language model

In [2]:
# temperature=0.3 gives mostly deterministic strategy selection and evaluation
# while allowing some variation in the actual reasoning output
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.3)

## The strategy catalogue
The five reasoning strategies cover the most common problem archetypes. Each strategy represents a different cognitive stance toward a problem: from the most direct (just answer it) through decomposition (separate parts), analogy (map from a known domain), abstraction (derive from principles), and systematic exploration (enumerate every relevant dimension). The catalogue is a dict because that is what it functionally is - a lookup table from strategy name to the prompt that instantiates that strategy. The `{problem}` placeholder in each template is filled in at call time by `.format(problem=problem)`.

In [3]:
# Each value is a prompt template; {problem} is substituted at call time.
# Keys are used as strategy names throughout the pipeline.
STRATEGIES = {
    "DIRECT": (
        "Solve this problem directly and concisely:\n\n{problem}"
    ),
    "DECOMPOSITION": (
        "Break this problem into smaller sub-problems, solve each one, "
        "then synthesise the parts into a complete answer:\n\n{problem}"
    ),
    "ANALOGICAL": (
        "Identify a familiar problem that is structurally similar to this one, "
        "explain the analogy clearly, then adapt that problem's solution to this context:\n\n{problem}"
    ),
    "STEP_BACK": (
        "First identify the fundamental principles or abstractions that are relevant to this problem. "
        "Then apply those principles step by step to derive a concrete solution:\n\n{problem}"
    ),
    "SYSTEMATIC": (
        "Systematically explore all key dimensions, constraints, and trade-offs involved. "
        "Consider every relevant angle before drawing a conclusion:\n\n{problem}"
    ),
}

print("Strategy catalogue:")
for name in STRATEGIES:
    print(f"  {name}")

Strategy catalogue:
  DIRECT
  DECOMPOSITION
  ANALOGICAL
  STEP_BACK
  SYSTEMATIC


`STRATEGIES` is the only shared data structure in the implementation. Every function that needs to know the valid strategy names iterates `STRATEGIES.keys()`, and every function that needs to apply a strategy calls `STRATEGIES[name].format(problem=problem)`. Adding a new strategy requires only one new dict entry; nothing else changes.

## Selecting a strategy
Strategy selection is the first meta-level act - the model is asked to reason about the problem before touching it. The prompt presents brief descriptions of all five strategies and asks the model to pick the one that best matches the problem's nature. The response is parsed by scanning for any of the strategy names; if none is found, `DIRECT` is used as a safe default. This is more robust than exact-match parsing because the model may include the strategy name in a sentence like "I would recommend the SYSTEMATIC approach."

In [4]:
def select_strategy(problem: str) -> str:
    """Ask the LLM which reasoning strategy best fits this problem.

    Returns one of the keys from STRATEGIES, defaulting to 'DIRECT' if parsing fails.
    """
    prompt = (
        f"Select the most appropriate reasoning strategy for the following problem.\n\n"
        f"Problem: {problem}\n\n"
        f"Available strategies:\n"
        f"  DIRECT        — straightforward reasoning for well-defined, simple problems\n"
        f"  DECOMPOSITION — break into sub-problems for complex, multi-part problems\n"
        f"  ANALOGICAL    — use a structural analogy for novel but familiar-shaped problems\n"
        f"  STEP_BACK     — identify abstract principles first for domain-knowledge-heavy problems\n"
        f"  SYSTEMATIC    — thorough exploration of all dimensions for open-ended problems\n\n"
        f"Briefly explain your reasoning, then end with:\n"
        f"STRATEGY: <strategy name>"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip().upper()   # uppercase for case-insensitive matching

    # Scan for any known strategy name in the response
    for name in STRATEGIES:
        if name in raw:
            return name

    return "DIRECT"   # safe fallback if the model's response cannot be parsed

The prompt asks the model to explain its reasoning before naming the strategy. This is intentional: asking for justification first tends to produce more considered choices than asking for the name alone, because the model must commit to a rationale before it can name the strategy. The scan over `STRATEGIES` iterates keys in insertion order, so the first matching name wins - for prompts that mention multiple strategies (e.g. "DECOMPOSITION is better than DIRECT here"), the one most prominently positioned in the response is usually found first.

In [5]:
# Test: choose a problem where the strategy selection should be non-trivial
test_problem = (
    "Design a system to reduce traffic congestion in a major city. "
    "Consider public transit, road capacity, driver behaviour, and environmental impact."
)
test_strategy = select_strategy(test_problem)

print(f"Problem: {test_problem}\n")
print(f"Selected strategy: {test_strategy}")

Problem: Design a system to reduce traffic congestion in a major city. Consider public transit, road capacity, driver behaviour, and environmental impact.

Selected strategy: DECOMPOSITION


## Reasoning with the selected strategy
With a strategy chosen, the object-level reasoning step applies it. The mechanism is simple: look up the strategy's prompt template from `STRATEGIES`, substitute the problem text into the `{problem}` placeholder, send the result to the LLM, and return its response. The strategy selection work done in the previous step is fully encoded in *which template gets chosen* — the reasoning step itself is a single LLM call with no additional logic.

In [6]:
def reason_with_strategy(problem: str, strategy: str) -> str:
    """Apply the named strategy to the problem and return the full reasoning output."""
    # Retrieve the template for this strategy; fall back to DIRECT if name is unknown
    template = STRATEGIES.get(strategy, STRATEGIES["DIRECT"])

    # Substitute the actual problem text into the {problem} placeholder
    prompt = template.format(problem=problem)

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

`STRATEGIES.get(strategy, STRATEGIES["DIRECT"])` provides a double safety net: it handles both an unknown strategy name (returns the `DIRECT` template) and the case where `strategy` is the key `"DIRECT"` (normal lookup). The `str.format` call raises a `KeyError` if the template contains a `{placeholder}` name that is not supplied - which cannot happen here since `{problem}` is always provided. The full output is returned without truncation; any downstream display can truncate as needed for readability.

In [7]:
# Apply the strategy selected in the previous test cell
test_reasoning = reason_with_strategy(test_problem, test_strategy)

print(f"Strategy: {test_strategy}\n")
print(f"Reasoning output:\n{test_reasoning[:400]}...")

Strategy: DECOMPOSITION

Reasoning output:
To design a system to reduce traffic congestion in a major city, we can break the problem into several smaller sub-problems. Each sub-problem will focus on a different aspect of the overall system. Here’s a structured approach:

### Sub-problem 1: Assessing Current Traffic Patterns
1. **Data Collection**: Gather data on current traffic patterns, including peak hours, traffic volume, and accident h...


## Evaluating the reasoning and deciding the next action
After the object-level reasoning step produces output, the meta-level evaluation step takes over. This is the heart of metacognitive prompting: the model is asked to assess its own work. The evaluation prompt shows the model what it just produced and asks two questions - was this complete and sufficient, and if not, which strategy would be better?

Crucially, the strategy switch (if any) is decided by the LLM, not by an arbitrary rotation. The original implementation, when recommending a strategy change, always selected the first non-current strategy from the enum - in practice always `DIRECT` - regardless of what the problem actually needed. Here, the model names the specific strategy it believes would better address the gap, and the driver validates that name against the catalogue before acting on it.

In [8]:
def evaluate_and_decide(problem: str, strategy: str, reasoning: str) -> Tuple[str, str]:
    # Build the menu of valid strategy names for the DECISION line
    strategy_names = ' | '.join(STRATEGIES.keys())

    prompt = (
        f"A '{strategy}' reasoning strategy was applied to the following problem. "
        f"Assess whether the result is complete and sufficient.\n\n"
        f"Problem: {problem}\n\n"
        f"Reasoning produced:\n{reasoning}\n\n"
        f"Respond with exactly two lines:\n"
        f"ASSESSMENT: <one sentence on completeness and quality>\n"
        f"DECISION: DONE | <strategy name if a different approach would help: {strategy_names}>"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    assessment = ""
    decision = "DONE"   # default: accept the reasoning as-is

    for line in raw.splitlines():
        line = line.strip()
        if line.upper().startswith('ASSESSMENT:'):
            assessment = line.split(':', 1)[1].strip()
        elif line.upper().startswith('DECISION:'):
            decision_text = line.split(':', 1)[1].strip().upper()
            if 'DONE' in decision_text:
                decision = 'DONE'
            else:
                # Extract whichever strategy name appears in the decision text
                for name in STRATEGIES:
                    if name in decision_text:
                        decision = name
                        break
                else:
                    decision = 'DONE'   # no valid name found: treat as done

    return assessment, decision   # two-element tuple unpacked by the driver

The function returns a two-element tuple `(assessment, decision)`. The driver unpacks it as `assessment, decision = evaluate_and_decide(...)`. The `decision` value is either the literal string `'DONE'` or one of the keys from `STRATEGIES`. The validation loop checks each key in turn; the `for-else` construct (the `else` block runs only when the loop finishes without `break`) handles the case where no valid strategy name appears in the model's decision text by falling back to `'DONE'` - a safe choice that avoids an infinite loop.

In [9]:
# Test with the reasoning produced in the previous cell
test_assessment, test_decision = evaluate_and_decide(
    test_problem, test_strategy, test_reasoning
)

print(f"Strategy used: {test_strategy}")
print(f"Assessment: {test_assessment}")
print(f"Decision:   {test_decision}")

Strategy used: DECOMPOSITION
Assessment: The reasoning is complete and of high quality, effectively addressing multiple facets of the traffic congestion problem.
Decision:   DONE


## The complete metacognitive pipeline
The three functions are now assembled into a driver that manages the meta-level control loop. The loop runs at most `max_iterations` times; in practice it usually terminates earlier because the evaluation step returns `DONE`. Each iteration records a `(strategy, reasoning, assessment)` triple in a `history` list. If the decision is to switch strategies, the driver validates the new name against `STRATEGIES` before adopting it - this guards against any parsing edge case producing an invalid strategy name. The final answer is taken from the last iteration's reasoning, since that is the most recent (and usually the most refined) output.

In [10]:
def metacognitive_reason(problem: str, max_iterations: int = 3) -> dict:
    print(f"Problem: {problem}\n")

    # ── Meta-level step 1: choose the reasoning strategy before solving ───
    print("Selecting reasoning strategy...")
    strategy = select_strategy(problem)
    print(f"  Selected: {strategy}\n")

    history = []   # accumulates (strategy, reasoning, assessment) per iteration

    for iteration in range(1, max_iterations + 1):
        print(f"Iteration {iteration} — Strategy: {strategy}")

        # ── Object-level: apply the strategy to produce reasoning ─────────
        reasoning = reason_with_strategy(problem, strategy)
        print(f"  Reasoning preview: {reasoning[:80]}...")

        # ── Meta-level step 2: evaluate the reasoning and decide next action
        assessment, decision = evaluate_and_decide(problem, strategy, reasoning)
        print(f"  Assessment: {assessment}")
        print(f"  Decision:   {decision}\n")

        # Record this iteration before potentially switching strategy
        history.append((strategy, reasoning, assessment))

        if decision == 'DONE':
            print(f"  Solution accepted after {iteration} iteration(s).")
            break

        # Validate the recommended switch before adopting it
        if decision in STRATEGIES:
            print(f"  Switching: {strategy} -> {decision}")
            strategy = decision   # adopt the LLM-recommended strategy
        else:
            break   # unknown decision: exit safely

    # The final answer is the reasoning from the last recorded iteration
    final_answer = history[-1][1]   # index [1] = reasoning field of the tuple

    return {
        "problem":      problem,
        "history":      history,        # List of (strategy, reasoning, assessment)
        "final_answer": final_answer,
        "iterations":   len(history),
    }

The `history` list is the reasoning audit trail. Each entry is a three-element tuple: the strategy used in that iteration, the full reasoning text it produced, and the assessment the meta-level evaluation gave it. A caller can inspect `result['history'][i][0]` for the strategy name, `[i][1]` for the full reasoning, and `[i][2]` for the assessment at each step. The driver does not have a separate `print_reasoning_process` method - all display logic lives in the calling code, which keeps the driver focused on producing the result rather than formatting it.

## Applying to a complex, multi-faceted problem
A good test for metacognitive prompting is a problem where the choice of strategy genuinely matters. A single-sentence factual question benefits little from SYSTEMATIC exploration. A design problem with multiple interacting constraints - like employee retention in a distributed organisation - is where strategy selection adds value: DECOMPOSITION might address each factor separately, STEP_BACK might derive principles of motivation theory first, and SYSTEMATIC might enumerate every lever available. The metacognitive system should select the right one and confirm it worked.

In [11]:
problem = (
    "How can we improve employee retention in a remote-first technology company "
    "experiencing high turnover rates among senior engineers?"
)

result = metacognitive_reason(problem, max_iterations=3)

print("\n" + "=" * 65)
print("REASONING HISTORY")
print("=" * 65)
for i, (strategy, reasoning, assessment) in enumerate(result["history"], 1):
    print(f"\n  Iteration {i}: {strategy}")
    print(f"  Assessment: {assessment}")

print("\n" + "=" * 65)
print("FINAL ANSWER")
print("=" * 65)
print(result["final_answer"])

Problem: How can we improve employee retention in a remote-first technology company experiencing high turnover rates among senior engineers?

Selecting reasoning strategy...
  Selected: DECOMPOSITION

Iteration 1 — Strategy: DECOMPOSITION
  Reasoning preview: To tackle the problem of improving employee retention in a remote-first technolo...
  Assessment: The reasoning is comprehensive and addresses key factors influencing employee retention effectively.
  Decision:   DONE

  Solution accepted after 1 iteration(s).

REASONING HISTORY

  Iteration 1: DECOMPOSITION
  Assessment: The reasoning is comprehensive and addresses key factors influencing employee retention effectively.

FINAL ANSWER
To tackle the problem of improving employee retention in a remote-first technology company facing high turnover rates among senior engineers, we can break it down into the following sub-problems:

### Sub-Problem 1: Identify the Causes of High Turnover
1. **Conduct Exit Interviews**: Analyze feedback

The `result['history']` loop unpacks each three-element tuple directly in the `for` statement - `for i, (strategy, reasoning, assessment) in enumerate(...)` - matching the structure in which the driver stored them. The full reasoning text at each iteration is available as `reasoning` but only the assessment is printed in this summary view; `result['final_answer']` contains the complete last-iteration output.

## Comparing with direct problem-solving
The comparison isolates the contribution of strategy selection and monitoring. The baseline sends the problem directly to the model with no meta-level framing, relying entirely on the model's default behaviour. The metacognitive approach first commits to a named strategy, applies it, and then checks whether the result is adequate. For well-defined problems the two approaches often produce similar answers; for open-ended or structurally complex problems the metacognitive version tends to be more structured and thorough - because the strategy prompt actively shapes the cognitive stance the model takes.

In [12]:
def direct_answer(problem: str) -> str:
    # Single call with no framing — the model chooses its own implicit approach
    response = llm.invoke([HumanMessage(
        content=f"Solve this problem: {problem}"
    )])
    return response.content.strip()


comparison_problem = (
    "A company must choose between three project proposals with different risk profiles, "
    "timelines, and expected returns. How should they make this decision?"
)

print("--- Direct answer ---")
direct = direct_answer(comparison_problem)
print(direct[:400] + "...")

print("\n" + "-" * 65)
print("--- Metacognitive prompting ---")
result = metacognitive_reason(comparison_problem, max_iterations=2)

print("\nStrategy history:")
for i, (strategy, _, assessment) in enumerate(result["history"], 1):
    print(f"  Iteration {i}: {strategy} — {assessment}")

print(f"\nFinal answer (first 400 chars):")
print(result["final_answer"][:400] + "...")

--- Direct answer ---
When a company is faced with multiple project proposals, it can use a structured decision-making process to evaluate and choose the best option. Here are the steps the company can follow:

1. **Define Objectives and Criteria**:
   - Clearly outline the objectives of the projects (e.g., maximizing returns, minimizing risk, aligning with strategic goals).
   - Establish evaluation criteria based on ...

-----------------------------------------------------------------
--- Metacognitive prompting ---
Problem: A company must choose between three project proposals with different risk profiles, timelines, and expected returns. How should they make this decision?

Selecting reasoning strategy...
  Selected: DIRECT

Iteration 1 — Strategy: DIRECT
  Reasoning preview: To make a decision between the three project proposals, the company should follo...
  Assessment: The reasoning is comprehensive and covers essential aspects of project evaluation, making it a robust decision

**When to use metacognitive prompting:** Problems where the choice of reasoning approach materially affects the quality of the answer - open-ended design problems, multi-stakeholder trade-off decisions, or novel problems where the right abstraction level is not obvious. For simple factual questions it adds overhead without benefit.

**Limitations:** Every iteration costs two LLM calls (reason + evaluate). For `max_iterations=3` that is up to seven calls total (one selection + three reason + three evaluate), compared to one call for baseline. The evaluation quality depends on the model's ability to assess its own output, which is imperfect - it may rate incomplete answers as sufficient or adequate answers as needing improvement.